## Creating 'Silver' database for data cleaning & transformation purposes.

In [0]:
CREATE DATABASE IF NOT EXISTS silver;

## Raw Data Null Values - Check

In [0]:
SELECT 
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(website_session_id) AS null_website_session_id,
  COUNT(*) - COUNT(created_at) AS null_created_at,
  COUNT(*) - COUNT(user_id) AS null_user_id,
  COUNT(*) - COUNT(is_repeat_session) AS null_is_repeat_session,
  COUNT(*) - COUNT(utm_source) AS null_utm_source,
  COUNT(*) - COUNT(utm_campaign) AS null_utm_campaign,
  COUNT(*) - COUNT(utm_content) AS null_utm_content,
  COUNT(*) - COUNT(device_type) AS null_device_type,
  COUNT(*) - COUNT(http_referer) AS null_http_referer
FROM bronze.website_sessions;


In [0]:
SELECT 
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(website_pageview_id) AS null_website_pageview_id,
  COUNT(*) - COUNT(created_at) AS null_created_at,
  COUNT(*) - COUNT(website_session_id) AS null_website_session_id,
  COUNT(*) - COUNT(pageview_url) AS null_pageview_url
FROM bronze.website_pageviews;


In [0]:
SELECT 
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(order_id) AS null_order_id,
  COUNT(*) - COUNT(created_at) AS null_created_at,
  COUNT(*) - COUNT(website_session_id) AS null_website_session_id,
  COUNT(*) - COUNT(user_id) AS null_user_id,
  COUNT(*) - COUNT(primary_product_id) AS null_primary_product_id,
  COUNT(*) - COUNT(items_purchased) AS null_items_purchased,
  COUNT(*) - COUNT(price_usd) AS null_price_usd,
  COUNT(*) - COUNT(cogs_usd) AS null_cogs_usd
FROM bronze.orders;


In [0]:
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(order_item_refund_id) AS null_order_item_id,
  COUNT(*) - COUNT(created_at) AS null_created_at,
  COUNT(*) - COUNT(order_item_id) AS null_order_item_id,
  COUNT(*) - COUNT(order_id) AS null_order_id,
  COUNT(*) - COUNT(refund_amount_usd) AS null_refund_amount_usd
FROM bronze.order_item_refunds;

In [0]:
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(order_item_id) AS null_order_item_id,
  COUNT(*) - COUNT(created_at) AS null_created_at,
  COUNT(*) - COUNT(order_id) AS null_order_id,
  COUNT(*) - COUNT(product_id) AS null_product_id,
  COUNT(*) - COUNT(is_primary_item) AS null_is_primary_item,
  COUNT(*) - COUNT(price_usd) AS null_price_usd,
  COUNT(*) - COUNT(cogs_usd) AS null_cogs_usd
FROM bronze.order_items;

In [0]:
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(product_id) AS null_product_id,
  COUNT(*) - COUNT(created_at) AS null_created_at
FROM bronze.products;

## Raw Data Blanks - Check

In [0]:
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN utm_source = '' THEN 1 ELSE 0 END) AS empty_utm_source,
    SUM(CASE WHEN utm_campaign = '' THEN 1 ELSE 0 END) AS empty_utm_campaign,
    SUM(CASE WHEN http_referer = '' THEN 1 ELSE 0 END) AS empty_referer
FROM bronze.website_sessions;

In [0]:
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN pageview_url = '' THEN 1 ELSE 0 END) AS empty_pageview_url
FROM bronze.website_pageviews;

In [0]:
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN product_name = '' THEN 1 ELSE 0 END) AS empty_product_name
FROM bronze.products;
  

## Datatype transformation and key logic implementation

### Website Sessions

In [0]:
CREATE OR REPLACE TABLE silver.website_sessions
USING DELTA
AS

SELECT 
  CAST(website_session_id AS BIGINT) AS website_session_id,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(user_id AS BIGINT) AS user_id,
  CAST(is_repeat_session AS INT) AS is_repeat_session,
  COALESCE(utm_source, 'direct traffic') AS utm_source,
  COALESCE(utm_campaign, 'organic') AS utm_campaign,
  COALESCE(utm_content, 'none') AS utm_content,
  device_type,
  http_referer,
  _ingested_at
FROM bronze.website_sessions

In [0]:
SELECT 
COUNT(*) AS total_rows,
COUNT(*) - COUNT(website_session_id) AS null_website_session_id,
COUNT(*) - COUNT(created_at) AS null_created_at,
COUNT(*) - COUNT(user_id) AS null_user_id,
COUNT(*) - COUNT(is_repeat_session) AS null_is_repeat_session,
COUNT(*) - COUNT(utm_source) AS null_utm_source,
COUNT(*) - COUNT(utm_campaign) AS null_utm_campaign,
COUNT(*) - COUNT(utm_content) AS null_utm_content,
COUNT(*) - COUNT(device_type) AS null_device_type,
COUNT(*) - COUNT(http_referer) AS null_http_referer
FROM silver.website_sessions;

In [0]:
SELECT 
  (SELECT COUNT(*) FROM bronze.website_sessions) AS total_rows_bronze,
  (SELECT COUNT(*) FROM silver.website_sessions) AS total_rows_silver,
  (SELECT COUNT(*) FROM bronze.website_sessions) - (SELECT COUNT(*) FROM silver.website_sessions) AS rows_diff

### Website Pageviews

In [0]:
CREATE OR REPLACE TABLE silver.website_pageviews
USING DELTA
AS

SELECT
CAST(website_pageview_id AS BIGINT) AS website_pageview_id,
CAST(created_at AS TIMESTAMP) AS created_at,
CAST(website_session_id AS BIGINT) AS website_session_id,

CASE
    WHEN pageview_url = '/home' THEN 'home'
    WHEN pageview_url = '/products' THEN 'products'
    WHEN pageview_url = '/the-original-mr-fuzzy' THEN 'mr_fuzzy'
    WHEN pageview_url = '/the-forever-love-bear' THEN 'love_bear'
    WHEN pageview_url = '/the-birthday-sugar-panda' THEN 'sugar_panda'
    WHEN pageview_url = '/the-hudson-river-mini-bear' THEN 'hudson_mini_bear'
    WHEN pageview_url = '/cart' THEN 'cart'
    WHEN pageview_url = '/shipping' THEN 'shipping'
    WHEN pageview_url = '/billing' THEN 'billing1'
    WHEN pageview_url = '/billing-2' THEN 'billing2'
    WHEN pageview_url = '/thank-you-for-your-order' THEN 'thank_you'
    WHEN pageview_url LIKE '%/lander-%' THEN 'lander'
    ELSE 'other'
END pageview_url,

CASE
    WHEN pageview_url = '/home' THEN 'homepage'
    WHEN pageview_url = '/products' THEN 'product_list'
    WHEN pageview_url = '/the-original-mr-fuzzy' THEN 'product_page'
    WHEN pageview_url = '/the-forever-love-bear' THEN 'product_page'
    WHEN pageview_url = '/the-birthday-sugar-panda' THEN 'product_page'
    WHEN pageview_url = '/the-hudson-river-mini-bear' THEN 'product_page'
    WHEN pageview_url = '/cart' THEN 'cart'
    WHEN pageview_url = '/shipping' THEN 'checkout'
    WHEN pageview_url = '/billing' THEN 'checkout'
    WHEN pageview_url = '/billing-2' THEN 'checkout'
    WHEN pageview_url = '/thank-you-for-your-order' THEN 'order_confirmation'
    WHEN pageview_url LIKE '%/lander-%' THEN 'promo_pages'
    ELSE 'other'
END category_url,

_ingested_at
FROM bronze.website_pageviews;

In [0]:
SELECT 
  (SELECT COUNT(*) FROM bronze.website_pageviews) AS total_rows_bronze,
  (SELECT COUNT(*) FROM silver.website_pageviews) AS total_rows_silver,
  (SELECT COUNT(*) FROM bronze.website_pageviews) - (SELECT COUNT(*) FROM silver.website_pageviews) AS rows_diff

In [0]:
SELECT 
COUNT(*) AS total_rows,
COUNT(*) - COUNT(website_pageview_id) AS null_website_pageview_id,
COUNT(*) - COUNT(created_at) AS null_created_at,
COUNT(*) - COUNT(website_session_id) AS null_website_session_id,
COUNT(*) - COUNT(pageview_url) AS null_pageview_url
FROM silver.website_pageviews;


### Orders

In [0]:
CREATE OR REPLACE TABLE silver.orders
USING DELTA
AS

SELECT
  CAST(order_id AS BIGINT) AS order_id,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(website_session_id AS BIGINT) AS website_session_id,
  CAST(user_id AS BIGINT) AS user_id,
  CAST(primary_product_id AS INT) AS primary_product_id,
  CAST(items_purchased AS INT) AS items_purchased,
  CAST(price_usd AS DECIMAL(10,2)) AS price_usd,
  CAST(cogs_usd AS DECIMAL(10,2)) AS cogs_usd,
  ROUND((price_usd - cogs_usd), 2) AS gross_margin,
  ROUND((price_usd - cogs_usd) / NULLIF(price_usd, 0) * 100, 2) AS gross_margin_percent,
  ROUND((price_usd * 0.23), 2) AS revenue_tax,
  _ingested_at
FROM bronze.orders;


In [0]:
SELECT 
  (SELECT COUNT(*) FROM bronze.orders) AS total_rows_bronze,
  (SELECT COUNT(*) FROM silver.orders) AS total_rows_silver,
  (SELECT COUNT(*) FROM bronze.orders) - (SELECT COUNT(*) FROM silver.orders) AS rows_diff

In [0]:
SELECT 
COUNT(*) AS total_rows,
COUNT(*) - COUNT(order_id) AS null_order_id,
COUNT(*) - COUNT(created_at) AS null_created_at,
COUNT(*) - COUNT(website_session_id) AS null_website_session_id,
COUNT(*) - COUNT(user_id) AS null_user_id,
COUNT(*) - COUNT(primary_product_id) AS null_primary_product_id,
COUNT(*) - COUNT(items_purchased) AS null_items_purchased,
COUNT(*) - COUNT(price_usd) AS null_price_usd,
COUNT(*) - COUNT(cogs_usd) AS null_cogs_usd
FROM silver.orders;


### Order Item Refunds

In [0]:
CREATE OR REPLACE TABLE silver.order_item_refunds
USING DELTA
AS

SELECT
  CAST(order_item_refund_id AS BIGINT) AS order_item_refund_id,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(order_item_id AS BIGINT) AS order_item_id,
  CAST(order_id AS BIGINT) AS order_id,
  CAST(refund_amount_usd AS DECIMAL(10,2)) AS refund_amount_usd,
  _ingested_at
FROM bronze.order_item_refunds

In [0]:
SELECT 
  (SELECT COUNT(*) FROM bronze.order_item_refunds) AS total_rows_bronze,
  (SELECT COUNT(*) FROM silver.order_item_refunds) AS total_rows_silver,
  (SELECT COUNT(*) FROM bronze.order_item_refunds) - (SELECT COUNT(*) FROM silver.order_item_refunds) AS rows_diff

In [0]:
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(order_item_refund_id) AS null_order_item_id,
  COUNT(*) - COUNT(created_at) AS null_created_at,
  COUNT(*) - COUNT(order_item_id) AS null_order_item_id,
  COUNT(*) - COUNT(order_id) AS null_order_id,
  COUNT(*) - COUNT(refund_amount_usd) AS null_refund_amount_usd
FROM silver.order_item_refunds;

### Order Items

In [0]:
CREATE OR REPLACE TABLE silver.order_items 
USING DELTA 
AS

SELECT
  CAST(order_item_id AS BIGINT) AS order_item_id,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(order_id AS BIGINT) AS order_id,
  CAST(product_id AS BIGINT) AS product_id,
  CAST(is_primary_item AS INT) AS is_primary_item,
  CAST(price_usd AS DECIMAL(10,2)) AS price_usd,
  CAST(cogs_usd AS DECIMAL(10,2)) AS cogs_usd,
  _ingested_at
FROM bronze.order_items;

In [0]:
SELECT 
  (SELECT COUNT(*) FROM bronze.order_items) AS total_rows_bronze,
  (SELECT COUNT(*) FROM silver.order_items) AS total_rows_silver,
  (SELECT COUNT(*) FROM bronze.order_items) - (SELECT COUNT(*) FROM silver.order_items) AS rows_diff

In [0]:
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(order_item_id) AS null_order_item_id,
  COUNT(*) - COUNT(created_at) AS null_created_at,
  COUNT(*) - COUNT(order_id) AS null_order_id,
  COUNT(*) - COUNT(product_id) AS null_product_id,
  COUNT(*) - COUNT(is_primary_item) AS null_is_primary_item,
  COUNT(*) - COUNT(price_usd) AS null_price_usd,
  COUNT(*) - COUNT(cogs_usd) AS null_cogs_usd
FROM silver.order_items;

### Products

In [0]:
CREATE OR REPLACE TABLE silver.products 
USING DELTA
AS

SELECT
  CAST(product_id AS BIGINT) AS product_id,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(product_name AS STRING) AS product_name,
  _ingested_at
FROM bronze.products;

In [0]:
SELECT 
  (SELECT COUNT(*) FROM bronze.products) AS total_rows_bronze,
  (SELECT COUNT(*) FROM silver.products) AS total_rows_silver,
  (SELECT COUNT(*) FROM bronze.products) - (SELECT COUNT(*) FROM silver.products) AS rows_diff

In [0]:
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(product_id) AS null_product_id,
  COUNT(*) - COUNT(created_at) AS null_created_at
FROM silver.products;